In [103]:
"""Document loading module for RAG system.

This module provides functionality to load support article data from JSON files
and convert it into formats suitable for RAG processing, including pandas
DataFrames and LangChain Document objects.
"""

from typing import List

import pandas as pd
import json
from pathlib import Path
from langchain_core.documents import Document


class DocumentLoader:
    """
    A utility class for loading and converting JSON support article data into LangChain Document objects.

    Input Requirement:
    - The input must be a JSON file with a list of support articles.
    - Each article must contain the following fields:
        - 'id': Unique identifier for each article.
        - 'title': The title of the support article.
        - 'content': The main content or body of the article.
        - 'category': Category of the article (e.g., 'integrations', 'billing', 'troubleshooting').
        - 'help_score': Numerical score indicating article helpfulness (0.0 to 1.0).
        - 'view_count': Number of times the article has been viewed.
        - 'last_updated': Timestamp indicating when the article was last updated.

    Preferred date format for 'last_updated': 'YYYY-MM-DD' or 'YYYY-MM-DD HH:MM:SS'.

    Example JSON format:
    [
        {
            "id": "KB001",
            "title": "How to integrate with Slack",
            "content": "To integrate our project management tool with Slack...",
            "category": "integrations",
            "help_score": 0.92,
            "view_count": 1523,
            "last_updated": "2024-10-05"
        }
    ]
    """

    def __init__(self, json_file: str):
        """
        Initializes the DocumentLoader with the path to a JSON file.

        Args:
            json_file (str): Absolute or relative path to the JSON file.

        Raises:
            ValueError: If input is not a non-empty string.
        """
        if not isinstance(json_file, str) or not json_file.strip():
            raise ValueError("json_file must be a non-empty string")

        self.json_file = json_file
        self.required_columns = {
            "id",
            "title",
            "content",
            "category",
            "help_score",
            "view_count",
            "last_updated",
        }

    def load_data(self) -> pd.DataFrame:
        """
        Loads and parses support article data from the specified JSON file.

        Functionality:
        - Verifies that the file exists and is accessible.
        - Parses the JSON content into a pandas DataFrame.
        - Validates that each article contains all required fields.
        - Returns a DataFrame with all article data.

        Returns:
            pd.DataFrame: A DataFrame containing all support articles with required columns.

        Raises:
            FileNotFoundError: If the file path does not exist.
            json.JSONDecodeError: If file contains invalid JSON.
            ValueError: If required fields are missing or DataFrame is empty.
            KeyError: If required columns are not present in the data.
        """

        file_path = Path(self.json_file)

        # ---------------------------------------------------------
        # 1. Validate file
        # ---------------------------------------------------------

        if not file_path.exists():
            raise FileNotFoundError(
                f"Data file not found: {file_path}"
            )

        if not file_path.is_file():
            raise ValueError(
                f"Data path is not a file: {file_path}"
            )

        # ---------------------------------------------------------
        # 2. Load JSON
        # ---------------------------------------------------------

        try:
            with file_path.open(
                mode="r",
                encoding="utf-8"
            ) as file:
                data = json.load(file)
        except PermissionError as exc:
            raise PermissionError(
                f"Permission denied when reading: {file_path}"
            ) from exc
        except json.JSONDecodeError as exc:
            raise json.JSONDecodeError(
                f"Invalid json in {file_path}: {exc.msg}",
                exc.doc,
                exc.pos
            ) from exc

        # ---------------------------------------------------------
        # 3. Validate JSON structure
        # ---------------------------------------------------------

        if not isinstance(data, list):
            raise ValueError(
                "JSON root must be a list of support articles"
            )

        if not data:
            raise ValueError(
                "JSON file contains no support articles"
            )

        if not all(isinstance(article, dict) for article in data):
            raise ValueError(
                "Each support article must be a JSON object"
            )

        # ---------------------------------------------------------
        # 4. Create DataFrame
        # ---------------------------------------------------------

        dataframe = pd.DataFrame(data)

        if dataframe.empty:
            raise ValueError(
                "DataFrame is empty after parsing JSON"
            )

        # ---------------------------------------------------------
        # 5. Validate columns
        # ---------------------------------------------------------

        missing_columns = self.required_columns.difference(
            dataframe.columns
        )

        if missing_columns:
            raise KeyError(
                "Required columns missing from dataframe: "
                + ", ".join(sorted(missing_columns))
            )

        # ---------------------------------------------------------
        # 6. Validate required fields
        # ---------------------------------------------------------
        null_columns = [
            column
            for column in self.required_columns
            if dataframe[column].isna().any()
        ]

        if null_columns:
            raise ValueError(
                "Required columns contain null values: "
                + ", ".join(sorted(null_columns))
            )

        # ---------------------------------------------------------
        # 7. Validate string fields
        # ---------------------------------------------------------

        string_columns = {
            "id",
            "title",
            "content",
            "category",
            "last_updated",
        }

        for column in string_columns:
            invalid_values = (
                ~dataframe[column].apply(
                    lambda value: isinstance(value, str)
                )
            )

            if invalid_values.any():
                raise ValueError(
                    f"Column '{column}' must contain only str"
                )

        # ---------------------------------------------------------
        # 8. Validate numeric fields
        # ---------------------------------------------------------

        numeric_columns = {
            "help_score",
            "view_count",
        }

        for column in numeric_columns:
            converted = pd.to_numeric(
                dataframe[column],
                errors="coerce",
            )

            if converted.isna().any():
                raise ValueError(
                    f"Column '{column}' must contain numeric values"
                )

            dataframe[column] = converted

        # ---------------------------------------------------------
        # 9. Validate numeric constraints
        # ---------------------------------------------------------

        if (dataframe["view_count"] < 0).any():
            raise ValueError(
                "'view_count' cannot contain negative values"
            )

        if (
            (dataframe["help_score"] < 0) |
            (dataframe["help_score"] > 1)
        ).any():
            raise ValueError(
                "'help_score' must be between 0 and 1"
            )

        # ---------------------------------------------------------
        # 10. Validate IDs
        # ---------------------------------------------------------

        if dataframe["id"].duplicated().any():
            duplicated_ids = (
                dataframe.loc[
                    dataframe["id"].duplicated(keep=False),
                    "id"
                ]
                .unique()
                .tolist()
            )

            raise ValueError(
                f"Duplicate article IDs found: {duplicated_ids}"
            )

        # ---------------------------------------------------------
        # 11. Normalize string values
        # ---------------------------------------------------------

        for column in string_columns:
            dataframe[column] = dataframe[column].str.strip()

        # ---------------------------------------------------------
        # 12. Final validation
        # ---------------------------------------------------------

        if dataframe["id"].eq("").any():
            raise ValueError(
                "'id' cannot contain empty values"
            )

        if dataframe["title"].eq("").any():
            raise ValueError(
                "'title' cannot contain empty values"
            )

        if dataframe["content"].eq("").any():
            raise ValueError(
                "'content' cannot contain empty values"
            )

        if dataframe["category"].eq("").any():
            raise ValueError(
                "'category' cannot contain empty values"
            )

        return dataframe

    def create_documents(self, data: pd.DataFrame) -> List[Document]:
        """
        Converts validated article data from a DataFrame into LangChain Document objects.

        Args:
            data (pandas.DataFrame): A DataFrame where each row represents a support article. Required columns:
                - 'id'
                - 'title'
                - 'content'
                - 'category'
                - 'help_score'
                - 'view_count'
                - 'last_updated'

        Returns:
            List[Document]: A list of LangChain-compatible Document objects with metadata.

        Raises:
            ValueError: If required columns are missing or DataFrame is empty.
            TypeError: If input is not a pandas DataFrame.
        """

        # ---------------------------------------------------------
        # 1. Validate input
        # ---------------------------------------------------------

        if not isinstance(data, pd.DataFrame):
            raise TypeError(
                "Input must be a pandas DataFrame"
            )

        # ---------------------------------------------------------
        # 2. Validate dataframe
        # ---------------------------------------------------------

        missing_columns = self.required_columns.difference(
            data.columns
        )

        if data.empty:
            raise ValueError(
                "DataFrame is empty"
            )

        if missing_columns:
            raise ValueError(
                "Required columns missing from dataframe: "
                + ", ".join(missing_columns)
            )

        # ---------------------------------------------------------
        # 3. Create documents
        # ---------------------------------------------------------

        documents = []
        for _, row in data.iterrows():
            doc = Document(
                page_content=row["content"],
                metadata={
                    "id": row["id"],
                    "title": row["title"],
                    "category": row["category"],
                    "help_score": row["help_score"],
                    "view_count": row["view_count"],
                    "last_updated": row["last_updated"]
                }
            )
            documents.append(doc)

        return documents

In [ ]:
"""Vector store module for semantic search in RAG system.

This module provides functionality to create, manage, and query vector embeddings
for semantic search. It handles embedding generation, index creation/loading,
and similarity search operations using sentence transformers.
"""

from typing import Dict, List, Tuple

import numpy as np
import pickle
import pandas as pd
from sentence_transformers import SentenceTransformer


class VectorStore:
    """
    A class for creating and managing vector embeddings and indexes for semantic search.

    Responsibilities:
    - Generate embeddings from article titles and content using SentenceTransformer models.
    - Build and persist vector indexes with metadata.
    - Load pre-built indexes from disk.
    - Perform efficient similarity search using cosine similarity.
    - Manage document metadata alongside embeddings.

    **Expected Fields in DataFrame:**
    The `df` parameter passed to `create_index()` should include:
    - 'id': Unique article identifier.
    - 'title': The title of the support article (used for embeddings).
    - 'content': The content of the article.
    - 'category': Article category (e.g., 'integrations', 'billing').
    - 'help_score': Helpfulness score (0.0 to 1.0).
    - 'view_count': Number of views.
    - 'last_updated': Timestamp of last update.

    **Embedding Generation:**
    Uses SentenceTransformer model `all-MiniLM-L6-v2` (384 dimensions) for embeddings.

    **Similarity Search:**
    Uses cosine similarity to find the most relevant documents for a query.
    """

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initializes the VectorStore with a specified embedding model.

        Args:
            model_name (str): Name of the SentenceTransformer model to use.
        """
        self.required_columns = {
            "id",
            "title",
            "content",
            "category",
            "help_score",
            "view_count",
            "last_updated",
        }

        self.model_name = model_name
        self.model = None

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generates sentence embeddings for a list of texts using SentenceTransformer.

        Args:
            texts (List[str]): A list of text strings (e.g., article titles or content).

        Returns:
            np.ndarray: Array of sentence embedding vectors (shape: [n_texts, embedding_dim]).

        Raises:
            ValueError: If `texts` is not a non-empty list.
            TypeError: If `texts` is not a list.
        """
        if not isinstance(texts, list) or not texts:
            raise ValueError("Input must be a non-empty list of strings")

        if not all(isinstance(text, str) for text in texts):
            raise TypeError("All elements in the input list must be strings")

        if self.model is None:
            self.model = SentenceTransformer(self.model_name)

        embeddings = self.model.encode(texts, convert_to_numpy=True)
        return embeddings

    def create_index(self, df: pd.DataFrame, index_file_name: str, index_folder_name: str) -> Dict:
        """
        Creates a vector index from the given DataFrame and saves it to disk.

        Process:
        1. Generate embeddings for article titles (or titles + content).
        2. Create index dictionary with embeddings and metadata.
        3. Save index to disk using pickle.
        4. Return the created index.

        Args:
            df (pd.DataFrame): DataFrame containing support articles with required fields:
                - 'id': Unique article ID.
                - 'title': Article title (used for embeddings).
                - 'content': Article content.
                - 'category': Article category.
                - 'help_score': Helpfulness score.
                - 'view_count': View count.
                - 'last_updated': Last updated timestamp.
            index_file_name (str): Name of the index file (e.g., 'support_index.pkl').
            index_folder_name (str): Directory where the index will be saved.

        Returns:
            dict: The created index containing embeddings and metadata.

        Raises:
            ValueError: If DataFrame is empty or missing required columns.
            OSError: If unable to create directory or save file.
        """
        if not isinstance(df, pd.DataFrame) or df.empty:
            raise ValueError("Input must be a non-empty DataFrame")

        missing_columns = self.required_columns.difference(df.columns)
        if missing_columns:
            raise ValueError(
                "DataFrame is missing required columns: "
                + ", ".join(missing_columns)
            )

        titles = df["title"].tolist()
        embeddings = self.generate_embeddings(titles)

        # Create index dictionary
        index = {
            "embeddings": embeddings,
        }

        for key in self.required_columns:
            index[key] = df[key].tolist()

        # Save index to disk
        index_dir = Path(index_folder_name)

        try:
            index_dir.mkdir(parents=True, exist_ok=True)
        except OSError as exc:
            raise OSError(
                f"Failed to create index directory: {index_dir}"
            ) from exc

        index_file_path = index_dir / index_file_name
        with open(index_file_path, "wb") as f:
            pickle.dump(index, f)

        return index

    def load_index(self, index_file_name: str, index_folder_name: str) -> Dict:
        """
        Loads a precomputed vector index from disk.

        Args:
            index_file_name (str): The file name of the saved index (e.g., 'support_index.pkl').
            index_folder_name (str): The directory where the index is stored.

        Returns:
            dict: The loaded index containing embeddings and metadata.

        Raises:
            FileNotFoundError: If the index file does not exist.
            ValueError: If the index structure is invalid or corrupted.
            KeyError: If required keys are missing in the index.
        """
        index_dir = Path(index_folder_name)
        index_file_path = index_dir / index_file_name

        try:
            with open(index_file_path, "rb") as f:
                index = pickle.load(f)
        except FileNotFoundError as exc:
            raise FileNotFoundError(
                f"Index file not found: {index_file_path}"
            ) from exc
        except (EOFError, ValueError) as exc:
            raise ValueError(
                f"Invalid index file format: {index_file_path}"
            ) from exc

        # Validate index structure
        if not isinstance(index, dict):
            raise ValueError(
                f"Invalid index structure in file: {index_file_path}"
            )

        # Check for required keys
        required_keys = {"embeddings", "id", "title", "content", "category", "help_score", "view_count", "last_updated"}
        missing_keys = required_keys.difference(index.keys())
        if missing_keys:
            raise KeyError(
                "Missing required keys in index: "
                + ", ".join(missing_keys)
            )

        return index

    def get_query_embedding(self, query: str) -> np.ndarray:
        """
        Generates an embedding for a single query string.

        Args:
            query (str): The query string to embed.

        Returns:
            np.ndarray: The embedding vector for the query (shape: [embedding_dim]).

        Raises:
            ValueError: If `query` is not a valid non-empty string.
        """
        if not isinstance(query, str) or not query.strip():
            raise ValueError("Query must be a non-empty string")

        if self.model is None:
            self.model = SentenceTransformer(self.model_name)

        query_embedding = self.model.encode(query, convert_to_numpy=True)
        return query_embedding

    def find_top_k_matches(self, query_embedding: np.ndarray, index: Dict, k: int = 5) -> List[Tuple[int, float]]:
        """
        Finds the top-k most similar articles from the index using cosine similarity.

        Args:
            query_embedding (np.ndarray): The embedding of the input query.
            index (dict): The index containing document embeddings and metadata.
            k (int): The number of top matches to return (default: 5).

        Returns:
            list: A list of tuples (document_index, similarity_score) representing the top-k matches,
                  sorted by similarity score in descending order.

        Raises:
            ValueError: If inputs are invalid (e.g., index missing embeddings).
            TypeError: If query_embedding is not a numpy array.
        """
        if not isinstance(query_embedding, np.ndarray):
            raise TypeError("query_embedding must be a numpy array")

        required_keys = {"embeddings", "id", "title", "content", "category", "help_score", "view_count", "last_updated"}

        if not required_keys.issubset(index.keys()):
            raise ValueError("Index must contain 'embeddings' and 'metadata' keys")

        embeddings = index["embeddings"]
        if not isinstance(embeddings, np.ndarray):
            raise ValueError("'embeddings' in index must be a numpy array")

        # Compute cosine similarity
        dot_product = np.dot(embeddings, query_embedding)
        norm_embeddings = np.linalg.norm(embeddings, axis=1)
        norm_query = np.linalg.norm(query_embedding)

        # Avoid division by zero
        if norm_query == 0:
            raise ValueError("Query embedding has zero norm")

        similarity_scores = dot_product / (norm_embeddings * norm_query)

        # Get top-k indices
        top_k_indices = np.argsort(similarity_scores)[-k:][::-1]
        top_k_scores = similarity_scores[top_k_indices]

        return list(zip(top_k_indices.tolist(), top_k_scores.tolist(), strict=True))

In [ ]:
"""Metadata filtering module for RAG system.

This module provides functionality to filter and rank support articles based on
metadata signals such as category matching, freshness, help scores, and view counts.
It combines semantic similarity scores with metadata signals for improved ranking.
"""

from datetime import datetime
from typing import Dict, List, Optional

import re
import numpy as np
import pandas as pd


class MetadataFilter:
    """
    Filters and ranks support articles based on metadata signals and relevance criteria.

    This class implements intelligent filtering and ranking strategies that consider:
    - Category matching (e.g., billing queries should return billing articles)
    - Article freshness (recent articles ranked higher)
    - Help score (articles with higher help scores ranked higher)
    - View count (popular articles may be more reliable)

    Attributes:
        category_weight (float): Weight for category matching (0.0–1.0).
        freshness_weight (float): Weight for article freshness (0.0–1.0).
        help_score_weight (float): Weight for help score (0.0–1.0).
    """

    def __init__(
        self,
        category_weight: float = 0.3,
        freshness_weight: float = 0.3,
        help_score_weight: float = 0.4
    ):
        """
        Initializes the MetadataFilter with configurable weights.

        Args:
            category_weight (float): Weight for category matching (default: 0.3).
            freshness_weight (float): Weight for article recency (default: 0.3).
            help_score_weight (float): Weight for help score (default: 0.4).

        Raises:
            TypeError: If any weight is not a float.
            ValueError: If any weight is not in range [0.0, 1.0].
        """
        if not isinstance(category_weight, (int, float)):
            raise TypeError("category_weight must be a float")
        if not isinstance(freshness_weight, (int, float)):
            raise TypeError("freshness_weight must be a float")
        if not isinstance(help_score_weight, (int, float)):
            raise TypeError("help_score_weight must be a float")

        if not 0.0 <= category_weight <= 1.0:
            raise ValueError("category_weight must be between 0.0 and 1.0")
        if not 0.0 <= freshness_weight <= 1.0:
            raise ValueError("freshness_weight must be between 0.0 and 1.0")
        if not 0.0 <= help_score_weight <= 1.0:
            raise ValueError("help_score_weight must be between 0.0 and 1.0")

        self.category_weight = float(category_weight)
        self.freshness_weight = float(freshness_weight)
        self.help_score_weight = float(help_score_weight)

    def calculate_freshness_score(
        self,
        last_updated: str,
        min_date: datetime,
        max_date: datetime
    ) -> float:
        """
        Calculates a freshness score based on an article's last updated date.

        More recent articles receive higher scores. Score is normalized between 0.0 and 1.0.

        Args:
            last_updated (str): Date string in 'YYYY-MM-DD' or 'YYYY-MM-DD HH:MM:SS' format.
            min_date (datetime): Earliest date in the dataset.
            max_date (datetime): Latest date in the dataset.

        Returns:
            float: Freshness score between 0.0 and 1.0 (higher = more recent).

        Notes:
            - If `last_updated` cannot be parsed, this function returns 0.0 instead of raising.
            - Uses pandas.to_datetime with errors='coerce' for robust parsing.
        """
        # Use pandas to robustly parse a variety of date string formats.
        parsed = pd.to_datetime(last_updated, errors='coerce')
        if pd.isna(parsed):
            # Unparseable date -> treat as oldest
            return 0.0

        # Ensure min_date and max_date are pandas Timestamps
        try:
            min_dt = pd.to_datetime(min_date)
            max_dt = pd.to_datetime(max_date)
        except Exception:
            # In case provided bounds are invalid, fallback to 0.0
            return 0.0

        # Compute timedeltas in seconds
        max_diff = (max_dt - min_dt).total_seconds()
        current_diff = (parsed - min_dt).total_seconds()

        # Safe handling when dataset has zero range
        if max_diff <= 0:
            return 0.0

        # Normalize and clamp between 0 and 1
        result = current_diff / max_diff
        return float(max(0.0, min(1.0, result)))

    def detect_query_category(self, query: str) -> Optional[str]:
        """Detects the most likely category for a query based on keyword matching.

        Categories:
        - 'integrations': Keywords like 'integrate', 'connect', 'API', 'webhook', 'Slack', 'Jira'
        - 'billing': Keywords like 'payment', 'invoice', 'subscription', 'pricing', 'charge'
        - 'troubleshooting': Keywords like 'error', 'bug', 'issue', 'not working', 'fix', 'problem'
        - 'getting-started': Keywords like 'setup', 'install', 'create', 'start', 'begin'
        - 'features': Keywords like 'how to', 'feature', 'functionality', 'use', 'customize'

        Args:
            query (str): The user's query string.

        Returns:
            Optional[str]: Detected category, or None if no clear match.

        Raises:
            TypeError: If query is not a string.
            ValueError: If query is empty or consists only of whitespace.
        """
        # 1. Validation checks
        if not isinstance(query, str):
            raise TypeError(f"Query must be of type 'str', got '{type(query).__name__}'")

        if not query.strip():
            raise ValueError("Query cannot be empty or consist only of whitespace")

        # 2. Normalize query to lowercase for case-insensitive matching
        query_lower = query.lower()

        # 3. Define categories with lowercase keywords
        categories = {
            'integrations': ['integrate', 'connect', 'api', 'webhook', 'slack', 'jira'],
            'billing': ['payment', 'invoice', 'subscription', 'pricing', 'charge'],
            'troubleshooting': ['error', 'bug', 'issue', 'not working', 'fix', 'problem'],
            'getting-started': ['setup', 'install', 'create', 'start', 'begin'],
            'features': ['how to', 'feature', 'functionality', 'use', 'customize']
        }

        best_category = None
        max_matches = 0

        # 4. Count matches utilizing word boundary regex to avoid partial-word matches
        for category, keywords in categories.items():
            count = 0
            for keyword in keywords:
                # Use raw strings with word boundaries (\b) to match entire words/phrases
                pattern = rf"\b{re.escape(keyword)}\b"
                if re.search(pattern, query_lower):
                    count += 1

            # Keep track of the category with the highest matches
            if count > max_matches:
                max_matches = count
                best_category = category

        # Returns None if no categories had at least 1 keyword match
        return best_category

    def calculate_category_score(self, article_category: str, detected_category: Optional[str]) -> float:
        """
        Calculates a category matching score.

        Args:
            article_category (str): The category of the article.
            detected_category (Optional[str]): The detected category from the query.

        Returns:
            float: 1.0 if categories match or no category detected, 0.0 otherwise.
        """
        if detected_category is None:
            return 1.0  # No detected category, treat as neutral match

        return 1.0 if article_category == detected_category else 0.0

    def calculate_metadata_score(
        self,
        help_score: float,
        freshness_score: float,
        category_score: float
    ) -> float:
        """
        Combines multiple metadata signals into a final score using weighted average.

        Args:
            help_score (float): The article's help score (0.0 to 1.0).
            freshness_score (float): The freshness score (0.0 to 1.0).
            category_score (float): The category match score (0.0 or 1.0).

        Returns:
            float: Combined metadata score.

        Raises:
            TypeError: If any score is not a float.
        """
        # 1. Validation checks
        if not all(isinstance(score, float) for score in [help_score, freshness_score, category_score]):
            raise TypeError("All scores must be of type 'float'")

        # 2. Weighted average calculation (weights can be adjusted as needed)
        return (
            self.help_score_weight * help_score +
            self.freshness_weight * freshness_score +
            self.category_weight * category_score
        )

    def filter_and_rank(
        self,
        df: pd.DataFrame,
        query: str,
        similarity_scores: List[float]
    ) -> pd.DataFrame:
        """
        Filters and ranks articles based on metadata and similarity scores.

        Process:
        1. Detect query category.
        2. Calculate freshness scores for all articles.
        3. Calculate category match scores.
        4. Combine metadata signals.
        5. Combine with similarity scores for final ranking.
        6. Return ranked DataFrame.

        Args:
            df (pd.DataFrame): DataFrame with retrieved articles containing:
                - 'id', 'title', 'content', 'category', 'help_score', 'view_count', 'last_updated'
            query (str): The user's query.
            similarity_scores (List[float]): Semantic similarity scores from vector search.

        Returns:
            pd.DataFrame: Ranked DataFrame with added 'metadata_score' and 'final_score' columns,
                          sorted by 'final_score' in descending order.

        Raises:
            ValueError: If DataFrame is empty or missing required columns.
            TypeError: If inputs have incorrect types.
        """
        # 1. Validation checks
        if df.empty:
            raise ValueError("DataFrame is empty")

        required_columns = {'id', 'title', 'content', 'category', 'help_score', 'view_count', 'last_updated'}
        if not required_columns.issubset(df.columns):
            raise ValueError(f"DataFrame is missing required columns: {required_columns - set(df.columns)}")

        if not isinstance(query, str):
            raise TypeError("Query must be a string")

        if not all(isinstance(score, float) for score in similarity_scores):
            raise TypeError("All similarity scores must be of type 'float'")

        # 2. Query category
        query_category = self.detect_query_category(query)

        # 3. Freshness scores for all articles
        min_date = pd.to_datetime(df['last_updated']).min()
        max_date = pd.to_datetime(df['last_updated']).max()
        freshness_scores = df['last_updated'].apply(lambda x: self.calculate_freshness_score(x, min_date, max_date))

        # 4. Category match scores
        category_scores = df['category'].apply(lambda x: self.calculate_category_score(x, query_category))

        # 5. Combine metadata signals
        df['metadata_score'] = df.apply(
            lambda row: self.calculate_metadata_score(
                help_score=row['help_score'],
                freshness_score=freshness_scores[row.name],
                category_score=category_scores[row.name]
            ),
            axis=1
        )

        # 6. Combine with similarity scores for final ranking
        df['final_score'] = df['metadata_score'] * 0.5 + pd.Series(similarity_scores) * 0.5

        # 7. Return ranked DataFrame
        return df.sort_values(by='final_score', ascending=False)


In [ ]:
"""Answer validation module for RAG system.

This module provides functionality to validate, generate, and process answers
generated by a RAG (Retrieval-Augmented Generation) system. It includes methods
for answer validation, generation using LLMs, and keyword extraction.
"""

from typing import Dict, List, Optional
import re
import json

from langchain_openai import ChatOpenAI


class AnswerValidator:
    """
    Validates generated answers for accuracy, completeness, and relevance.

    This class evaluates answers generated by RAG systems to ensure they meet quality standards:
    - Accuracy: Answer correctly addresses the query
    - Completeness: Answer contains all necessary information
    - Relevance: Answer is directly related to the query
    - Hallucination Detection: Identifies when answer contains unsupported information
    - Keyword Matching: Extracts and matches keywords between query and answer

    The validator can trigger re-retrieval when answer quality is insufficient.
    """

    def __init__(self, model_name: str = "gpt-4o-mini", temperature: float = 0.0):
        """
        Initializes the AnswerValidator with an LLM for evaluation.

        Args:
            model_name (str): Name of the OpenAI model to use for validation.
            temperature (float): Temperature for the LLM (default: 0.0 for deterministic output).

        Attributes:
            llm: The ChatOpenAI instance used for answer evaluation.
            validation_chain: Optional validation chain (can be set later).
        """
        self.llm = ChatOpenAI(model=model_name, temperature=temperature)
        self.validation_chain = None

    def validate_answer(self, query: str, answer: str) -> Dict:
        """
        Validate a generated answer against the original query.
        """
        if not isinstance(query, str):
            raise TypeError("Query must be a string")

        if not query.strip():
            raise ValueError("Query cannot be empty")

        if not isinstance(answer, str):
            raise TypeError("Answer must be a string")

        if not answer.strip():
            raise ValueError("Answer cannot be empty")

        query_keywords = self.extract_keywords(query)
        answer_keywords = self.extract_keywords(answer)

        answer_keyword_set = set(answer_keywords)

        keyword_overlap = [
            keyword
            for keyword in query_keywords
            if keyword in answer_keyword_set
        ]

        unique_query_keywords = set(query_keywords)

        keyword_score = (
            len(keyword_overlap) / len(unique_query_keywords)
            if unique_query_keywords
            else 0.0
        )

        # Deterministic quality checks. These are also useful when the LLM
        # response is mocked or unavailable.
        is_complete = len(answer.split()) >= 5
        has_vague_language = self._has_vague_language(answer)

        # A reasonable keyword match plus a substantive answer is considered
        # moderately relevant even if LLM validation is unavailable.
        fallback_score = keyword_score

        if len(keyword_overlap) >= 2 and is_complete and not has_vague_language:
            fallback_score = max(fallback_score, 0.6)

        validation_prompt = f"""
        You are an answer-quality evaluator for a RAG system.

        Evaluate the following answer against the user's query.

        Query:
        {query}

        Answer:
        {answer}

        Return ONLY valid JSON:

        {{
            "relevance_score": 0.0,
            "is_complete": true,
            "has_vague_language": false,
            "requires_requery": false
        }}

        Rules:
        - relevance_score must be between 0.0 and 1.0.
        - is_complete is true when the answer sufficiently addresses the query.
        - has_vague_language is true when the answer is unclear or evasive.
        - requires_requery is true when the answer is substantially irrelevant,
        incomplete, vague, or insufficient.
        """

        try:
            response = self.llm.invoke(validation_prompt)
            raw_content = response.content

            if not isinstance(raw_content, str):
                raw_content = str(raw_content)

            validation = self._parse_validation_response(raw_content)

            relevance_score = self._clamp_score(
                validation.get("relevance_score", fallback_score)
            )
            is_complete = bool(
                validation.get("is_complete", is_complete)
            )
            has_vague_language = bool(
                validation.get(
                    "has_vague_language",
                    has_vague_language,
                )
            )

            requires_requery = bool(
                validation.get("requires_requery", False)
                or relevance_score < 0.5
                or not is_complete
                or has_vague_language
            )

        except Exception:
            relevance_score = fallback_score
            requires_requery = bool(
                relevance_score < 0.5
                or not is_complete
                or has_vague_language
            )

        return {
            "relevance_score": relevance_score,
            "is_complete": is_complete,
            "has_vague_language": has_vague_language,
            "keyword_overlap": keyword_overlap,
            "requires_requery": requires_requery,
        }


    def generate_answer(
        self,
        query: str,
        documents: List[str],
        category: Optional[str] = None,
    ) -> str:
        """
        Generate an answer using retrieved documents.
        """
        if not isinstance(query, str):
            raise TypeError("Query must be a string")

        if not query.strip():
            raise ValueError("Query cannot be empty")

        if not isinstance(documents, list):
            raise TypeError("Documents must be a list of strings")

        if not documents:
            raise ValueError("Documents list cannot be empty")

        if not all(isinstance(document, str) for document in documents):
            raise TypeError("Every document must be a string")

        valid_documents = [
            document.strip()
            for document in documents
            if document.strip()
        ]

        if not valid_documents:
            return "I don't have enough information in the retrieved documents to answer this question."

        context = "\n\n--- DOCUMENT ---\n\n".join(valid_documents)

        category_context = (
            f"\nCategory: {category.strip()}\n"
            if category and category.strip()
            else ""
        )

        prompt = f"""
        You are an assistant answering questions using retrieved documents
        from a RAG system.

        Use the retrieved documents as the source of truth.
        Do not invent information that is not supported by them.

        If the documents do not contain enough information to answer the question,
        respond exactly with:

        I don't have enough information in the retrieved documents to answer this question.

        {category_context}

        User question:
        {query}

        Retrieved documents:
        {context}

        Answer clearly and concisely.
        """

        try:
            response = self.llm.invoke(prompt)
            content = response.content

            # This is important for mocked LLMs used in tests.
            if not isinstance(content, str):
                content = ""

            content = content.strip()

            if content:
                return content

        except Exception:
            pass

        return "I don't have enough information in the retrieved documents to answer this question."


    def extract_keywords(self, text: str) -> List[str]:
        """
        Extract unique, meaningful keywords from text.

        Keywords are normalized to lowercase, punctuation is removed,
        stop words are excluded, and duplicates are removed while
        preserving their original order.
        """
        if not isinstance(text, str):
            raise TypeError("Text must be a string")

        if not text.strip():
            raise ValueError("Text cannot be empty or whitespace")

        tokens = re.findall(r"\b[\w'-]+\b", text.lower())

        stop_words = {
            "a",
            "about",
            "after",
            "all",
            "also",
            "an",
            "and",
            "any",
            "are",
            "as",
            "at",
            "be",
            "because",
            "been",
            "before",
            "being",
            "between",
            "both",
            "but",
            "by",
            "can",
            "could",
            "did",
            "do",
            "does",
            "doing",
            "down",
            "during",
            "each",
            "for",
            "from",
            "further",
            "had",
            "has",
            "have",
            "having",
            "he",
            "her",
            "here",
            "hers",
            "him",
            "his",
            "how",
            "i",
            "if",
            "in",
            "into",
            "is",
            "it",
            "its",
            "itself",
            "just",
            "me",
            "more",
            "most",
            "my",
            "myself",
            "no",
            "nor",
            "not",
            "of",
            "off",
            "on",
            "once",
            "only",
            "or",
            "other",
            "our",
            "ours",
            "out",
            "over",
            "own",
            "same",
            "she",
            "should",
            "so",
            "some",
            "such",
            "than",
            "that",
            "the",
            "their",
            "theirs",
            "them",
            "then",
            "there",
            "these",
            "they",
            "this",
            "those",
            "through",
            "to",
            "too",
            "under",
            "until",
            "up",
            "very",
            "was",
            "we",
            "were",
            "what",
            "when",
            "where",
            "which",
            "while",
            "who",
            "whom",
            "why",
            "will",
            "with",
            "would",
            "you",
            "your",
            "yours",
        }

        # dict.fromkeys() removes duplicates while preserving order.
        keywords = [
            token
            for token in tokens
            if token not in stop_words and len(token) > 1
        ]

        return list(dict.fromkeys(keywords))

    @staticmethod
    def _clamp_score(value: object) -> float:
        """Convert a value to a float score between 0 and 1."""
        try:
            score = float(value)
        except (TypeError, ValueError):
            return 0.0

        return max(0.0, min(1.0, score))

    @staticmethod
    def _has_vague_language(answer: str) -> bool:
        """Detect common vague or evasive phrases."""
        vague_patterns = [
            r"\bit depends\b",
            r"\bmaybe\b",
            r"\bpossibly\b",
            r"\bsometimes\b",
            r"\bkind of\b",
            r"\bsort of\b",
            r"\bprobably\b",
            r"\bperhaps\b",
            r"\bnot sure\b",
            r"\bI don't know\b",
            r"\bI am not sure\b",
        ]

        return any(
            re.search(pattern, answer, flags=re.IGNORECASE)
            for pattern in vague_patterns
        )

    @staticmethod
    def _parse_validation_response(content: str) -> Dict:
        """
        Parse JSON returned by the validation LLM.

        Handles responses wrapped in Markdown code fences.
        """
        content = content.strip()
        content = re.sub(
            r"^```(?:json)?\s*|\s*```$",
            "",
            content,
            flags=re.IGNORECASE,
        ).strip()

        try:
            parsed = json.loads(content)
        except json.JSONDecodeError as exc:
            raise ValueError(
                "LLM validation response was not valid JSON"
            ) from exc

        if not isinstance(parsed, dict):
            raise ValueError("LLM validation response must be a JSON object")

        return parsed

In [ ]:
"""RAG pipeline module for complete retrieval-augmented generation system.

This module integrates all components (document loading, vector search, metadata filtering,
answer generation, and validation) into a complete RAG pipeline.
"""

from typing import Dict, List, Any, Optional

# from document_loader import DocumentLoader
# from vector_store import VectorStore
# from metadata_filter import MetadataFilter
# from answer_validator import AnswerValidator


class RAGPipeline:
    """
    A complete RAG pipeline that orchestrates document processing, retrieval, filtering,
    answer generation, and validation.

    **Pipeline Flow:**
    1. Load documents from JSON file using DocumentLoader
    2. Build vector index using VectorStore
    3. Retrieve relevant documents using VectorStore.find_top_k_matches()
    4. Filter and rank using MetadataFilter.filter_and_rank()
    5. Generate answer using AnswerValidator.generate_answer() (LLM call)
    6. Validate answer using AnswerValidator.validate_answer() (LLM call)
    7. Return structured response with answer, sources, confidence, and validation metrics
    """

    def __init__(
        self,
        document_loader: DocumentLoader,
        vector_store: VectorStore,
        metadata_filter: MetadataFilter,
        answer_validator: AnswerValidator
    ):
        """
        Initializes the RAG pipeline with all required components.

        Args:
            document_loader (DocumentLoader): Loader for reading documents from JSON.
            vector_store (VectorStore): Store for generating embeddings and dense retrieval.
            metadata_filter (MetadataFilter): Filter for ranking documents by metadata.
            answer_validator (AnswerValidator): Validator for generating and validating answers.
        """
        self.document_loader: DocumentLoader = document_loader
        self.vector_store: VectorStore = vector_store
        self.metadata_filter: MetadataFilter = metadata_filter
        self.answer_validator: AnswerValidator = answer_validator

        self.df = None
        self.vector_index = None

    def build_index(self, index_file_name: str = "support_index.pkl", index_folder_name: str = "index") -> Dict:
        """
        Builds the complete index for the RAG system.

        Process:
        1. Load documents from JSON file using DocumentLoader
        2. Build vector index using VectorStore
        3. Persist index to disk
        4. Return the created index

        Args:
            index_file_name (str): Name for the saved index file (default: "support_index.pkl").
            index_folder_name (str): Directory to save the index (default: "index").

        Returns:
            dict: The created vector index.

        Raises:
            ValueError: If documents cannot be loaded.
            OSError: If index cannot be saved.
        """
        try:
            self.df = self.document_loader.load_data()
            self.vector_index = self.vector_store.create_index(
                df=self.df,
                index_file_name=index_file_name,
                index_folder_name=index_folder_name
            )
            return self.vector_index
        except Exception as e:
            raise ValueError(f"Error building index: {e}")

    def load_index(self, index_file_name: str = "support_index.pkl", index_folder_name: str = "index") -> Dict:
        """
        Loads a pre-built index from disk.

        Args:
            index_file_name (str): Name of the index file to load.
            index_folder_name (str): Directory where the index is stored.

        Returns:
            dict: The loaded vector index.

        Raises:
            FileNotFoundError: If index file does not exist.
        """
        try:
            self.vector_index = self.vector_store.load_index(
                index_file_name,
                index_folder_name
            )
            return self.vector_index
        except FileNotFoundError as e:
            raise FileNotFoundError(f"Index file not found: {e}")

    def query(
        self,
        question: str,
        top_k: int = 5
    ) -> Dict[str, Any]:
        """
        Processes a query and returns a response with relevant context.

        Process:
        1. Retrieve top-k documents using VectorStore.find_top_k_matches()
        2. Filter and rank using MetadataFilter.filter_and_rank()
        3. Generate answer using AnswerValidator.generate_answer() (LLM call)
        4. Validate answer using AnswerValidator.validate_answer() (LLM call)
        5. Return structured response

        Args:
            question (str): The user's question or query.
            top_k (int): Number of top documents to retrieve (default: 5).

        Returns:
            dict: Response dictionary containing:
                - 'answer': Generated answer text
                - 'sources': List of source article IDs
                - 'confidence': Confidence level ("high", "medium", or "low")
                - 'validation': Validation results dict with relevance_score, is_complete, etc.

        Raises:
            ValueError: If question is empty or index not built.
        """
        if not question.strip():
            raise ValueError("question cannot be empty")

        if not self.vector_index:
            raise ValueError("Index not built.")

        if self.df is None or self.df.empty:
            raise ValueError("DataFrame not loaded.")

        # Step 0: Get query embedding
        query_embedding = self.vector_store.get_query_embedding(question)

        # Step 1: Retrieve top-k documents
        top_k_docs = self.vector_store.find_top_k_matches(
            query_embedding=query_embedding,
            k=top_k
        )

        # Step 2: Filter and rank using metadata
        filtered_docs = self.metadata_filter.filter_and_rank(
            documents=top_k_docs,
            query=question
        )

        # Step 3: Generate answer using LLM
        generated_answer = self.answer_validator.generate_answer(
            query=question,
            documents=filtered_docs
        )

        # Step 4: Validate answer using LLM
        validation_results = self.answer_validator.validate_answer(
            query=question,
            answer=generated_answer
        )

        # Step 5: Return structured response
        response = {
            "answer": generated_answer,
            "sources": [doc[0] for doc in filtered_docs],
            "confidence": self._calculate_confidence(validation_results),
            "validation": validation_results
        }

        return response

    def _calculate_confidence(self, validation: Dict[str, Any]) -> str:
        score = validation.get("relevance_score", 0.0)

        if score >= 0.8:
            return "high"
        elif score >= 0.5:
            return "medium"
        return "low"